# 06b-b — Forensic causale del coupling voltaggio/STATE

Congela i sei updater STATE del 06b e addestra soltanto un piccolo ponte causale per stimare `Delta V` a 1 ms. Il controllo shuffled usa le stesse predizioni ma ne distrugge l'allineamento causale. È un test di componente train-only, non un rollout autonomo del neurone.

In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main');ROOT=Path('/kaggle/working');ELM_REPO=ROOT/'hayflow_workspace'/'elmneuron';ELM_REPO.parent.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO);REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches();print({'revision':REVISION})

## 1. Autorità e dataset

Caricare come Input Kaggle il risultato 06b, l'autorità 05t, il dataset targeted base e il top-up BAP v3. Artefatti e checkpoint vengono selezionati e verificati tramite SHA-256, non tramite il nome assegnato da Kaggle.

In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import discover_indexed_artifact_source
from src.hayflow_model.atomic_state_dynamics_playground import EXPECTED_05T_INDEX_SHA256
from src.hayflow_model.causal_voltage_state_coupling_forensic import EXPECTED_06B_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input')
source05t=os.environ.get('HAYFLOW_05T_ARTIFACT');ARTIFACT_05T_SOURCE=discover_indexed_artifact_source(INPUT_ROOT,EXPECTED_05T_INDEX_SHA256,override=Path(source05t) if source05t else None);assert ARTIFACT_05T_SOURCE is not None,'Artefatto 05t esatto non trovato.'
source06b=os.environ.get('HAYFLOW_06B_ARTIFACT');ARTIFACT_06B_SOURCE=discover_indexed_artifact_source(INPUT_ROOT,EXPECTED_06B_INDEX_SHA256,override=Path(source06b) if source06b else None);assert ARTIFACT_06B_SOURCE is not None,'Artefatto 06b esatto non trovato: aggiungi hayflow_optimized_explicit_state_updater_canary agli Input Kaggle.'
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_stamp';stamp=f'{source.stat().st_size}:{source.stat().st_mtime_ns}'
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow06bb_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.';print({'05t':str(ARTIFACT_05T_SOURCE),'06b':str(ARTIFACT_06B_SOURCE),'base':str(BASE_SOURCE),'composite_manifest':str(COMPOSITE_MANIFEST)})

In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 06b-b][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880 and not bundle.manifest['physical_merge_performed'];print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})

## 2. Preflight e congelamento

Verifica il risultato 06b, ricostruisce i ruoli train-only e carica i sei checkpoint STATE in modalità frozen. Nessuna microtraccia futura viene letta; il solo componente trainabile è il ponte di voltaggio.

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import CausalVoltageStateCouplingConfig,CausalVoltageStateCouplingForensic
cfg=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_causal_voltage_state_coupling_forensic.yml').read_text());config=CausalVoltageStateCouplingConfig.from_mapping(cfg['causal_voltage_state_coupling_forensic'])
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_causal_voltage_state_coupling_forensic');assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
session=CausalVoltageStateCouplingForensic(bundle,OUTPUT_DIR,config,ARTIFACT_05T_SOURCE,ARTIFACT_06B_SOURCE,code_revision=REVISION);preflight=session.prepare_coupling_forensic();display({'valid':preflight['valid'],'roles':preflight['role_transition_counts'],'frozen_state_checkpoints':preflight['frozen_state_checkpoint_count'],'trainable_component':preflight['trainable_component'],'bridge_parameters':preflight['bridge_parameter_count'],'bridge_ceiling':preflight['bridge_parameter_ceiling'],'modes':preflight['coupling_modes'],'future_microtraces_read':preflight['future_microtraces_read'],'recursive_quantity':preflight['recursive_quantity'],'voltage_boundary':preflight['voltage_boundary_condition']});assert preflight['valid'] and preflight['frozen_state_checkpoint_count']==6 and preflight['state_and_outcome_splits_read']==['train'] and not preflight['future_microtraces_read'] and not preflight['validation_state_accessed'] and not preflight['test_state_accessed'] and not preflight['autonomous_voltage_rollout_claimed']

## 3. Ponti ΔV e controlli causali

Addestra tre ponti appaiati (uno per seed), quindi confronta causal, predicted, shuffled e oracle. Il tracker stampa solo righe compatte ogni 100 step; non visualizza tensori o dizionari estesi.

In [ ]:
bridge_report=session.train_voltage_bridges();summary={seed:{'calibration_V_rmse_mv':round(row['best_calibration_voltage_delta_rmse_mv'],4),'development_V_gain':round(row['development']['improvement_vs_persistence_fraction'],4),'active_V_gain':round(row['development']['active_improvement_vs_persistence_fraction'],4)} for seed,row in bridge_report['runs'].items()};display({'valid':bridge_report['valid'],'device':bridge_report['device'],'bridge_parameter_count':bridge_report['bridge_parameter_count'],'state_updater_retraining':bridge_report['state_updater_retraining_performed'],'runs':summary});assert bridge_report['valid'] and not bridge_report['state_updater_retraining_performed']

In [ ]:
try:
 one_step_report=session.evaluate_one_step_coupling();rollout_report=session.evaluate_coupled_nested_rollouts();final_report=session.finalize_coupling_forensic(bridge_report,one_step_report,rollout_report)
finally:
 session.close()
compact={seed:{'V_gain':round(row['voltage_gain'],4),'state_causal':round(row['state_gains']['frozen_causal'],4),'state_predicted':round(row['state_gains']['predicted_endpoint'],4),'state_shuffled':round(row['state_gains']['shuffled_predicted_endpoint'],4),'state_oracle':round(row['state_gains']['teacher_endpoint_oracle'],4),'predicted_over_causal':round(row['predicted_gain_over_causal'],4),'predicted_over_shuffled':round(row['predicted_gain_over_shuffled'],4),'oracle_gap_recovery':round(row['oracle_gap_recovery'],4),'predicted_over_causal_8ms':round(row['eight_ms_predicted_gain_over_causal'],4)} for seed,row in final_report['per_seed'].items()}
display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'coupling_identified':final_report['coupling_identified'],'gates':final_report['gate_checks'],'median':{k:round(v,4) for k,v in final_report['median'].items()},'per_seed':compact,'recursive_quantity':final_report['recursive_quantity'],'voltage_boundary':final_report['voltage_boundary_condition'],'next_step':final_report['next_step']});assert final_report['valid'] and final_report['component_decision_grade'] and not final_report['state_updater_retraining_performed'] and not final_report['autonomous_voltage_rollout_claimed'] and not final_report['validation_state_accessed'] and not final_report['test_state_accessed']
from IPython.display import Image;display(Image(filename=str(OUTPUT_DIR/'figures/causal_voltage_state_coupling.png')))

## 4. Crea e scarica lo ZIP

Downloader browser stabile del progetto: crea lo ZIP in `/kaggle/working`, lo converte in base64, ricostruisce un `Blob` e avvia il click temporaneo.

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_causal_voltage_state_coupling_forensic','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})